# Reactor.RSS Version v4.0.1
## Get Your Atom Packages as an RSS News Feed
reactormagic.png

In [ ]:
import requests, json, re, io, html
import xml.etree.ElementTree as ET
from datetime import datetime, timezone, timedelta
from xml.dom import minidom
from google.colab import files

HIGHLIGHTS_URL = "https://gitlab.com/WeSuckLess/Reactor/-/raw/master/JSON/Highlights.json?ref_type=heads"

RepoJSON = """
{
  "Repos": [
    {
      "Name": "Fusion",
      "JSON": "https://gitlab.com/WeSuckLess/Reactor/-/raw/master/JSON/Reactor.json?ref_type=heads"
    },
    {
      "Name": "Assimilate",
      "JSON": "https://gitlab.com/WeSuckLess/Reactor-for-Assimilate/-/raw/main/JSON/Reactor.json?ref_type=heads"
    },
    {
      "Name": "Houdini",
      "JSON": "https://gitlab.com/WeSuckLess/Reactor-for-Houdini/-/raw/main/JSON/Reactor.json?ref_type=heads"
    },
    {
      "Name": "LightWave3D",
      "JSON": "https://gitlab.com/WeSuckLess/Reactor-for-LightWave/-/raw/main/JSON/Reactor.json?ref_type=heads"
    },
    {
      "Name": "JangaFX",
      "JSON": "https://gitlab.com/WeSuckLess/Reactor-for-JangaFX/-/raw/main/JSON/Reactor.json?ref_type=heads"
    },
    {
      "Name": "OpenFX",
      "JSON": "https://gitlab.com/WeSuckLess/Reactor-for-OpenFX/-/raw/main/JSON/Reactor.json?ref_type=heads"
    },
    {
      "Name": "Pipelines",
      "JSON": "https://gitlab.com/WeSuckLess/Reactor-for-Pipelines/-/raw/main/JSON/Reactor.json?ref_type=heads"
    }
  ]
}
"""

# What should the RSS feed be called that is generated?
OUTPUT_FILE    = "feed.xml"

# Include up to 1500 packages.
# Note: This variable acts as a clamp on the output if there are tons and tons of packages.
MAX_PACKAGES   = 1500

# Read the JSON data
def fetch_json(url):
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return r.json()

print(" Fetching Highlights.json…")
highlights = fetch_json(HIGHLIGHTS_URL)
print(f"  {len(highlights)} highlights loaded")

# Format the HTML Content
def process_description(text, limit=None):
    if not text:
        return ""
    unescaped_text = html.unescape(text)
    clean_text = re.sub(r"<img[^>]*>", "", unescaped_text)
    if limit is not None:
        return clean_text[:limit]
    return clean_text

# Reactor JSON Date Records
def parse_reactor_date(date_field):
    try:
        y = int(date_field.get("1.0", 2018))
        m = int(date_field.get("2.0", 1))
        d = int(date_field.get("3.0", 21))
        return datetime(y, m, d, tzinfo=timezone.utc)
    except Exception:
        return datetime.now(timezone.utc)

def rfc822(dt):
    return dt.strftime("%a, %d %b %Y %H:%M:%S +0000")

# RSS parameters
ET.register_namespace("sy", "http://purl.org/rss/1.0/modules/syndication/")
ET.register_namespace("media", "http://search.yahoo.com/mrss/")
ET.register_namespace("dc", "http://purl.org/dc/elements/1.1/")
rss = ET.Element("rss", version="2.0")
channel = ET.SubElement(rss, "channel")
ET.SubElement(channel, "title").text = "Reactor Package Manager"
ET.SubElement(channel, "link").text  = "https://gitlab.com/WeSuckLess/Reactor"
ET.SubElement(channel, "description").text = (
    "Highlights and latest atom package releases from the Reactor Package Manager."
)
ET.SubElement(channel, "language").text = "en-us"
ET.SubElement(channel, "lastBuildDate").text = rfc822(datetime.now(timezone.utc))

# Refresh the feed every 10 minutes
ET.SubElement(channel, "{http://purl.org/rss/1.0/modules/syndication/}updatePeriod").text = "hourly"
ET.SubElement(channel, "{http://purl.org/rss/1.0/modules/syndication/}updateFrequency").text = "6"

now = datetime.now(timezone.utc)

# Highlights JSON
for idx, key in enumerate(highlights.keys()):
    h = highlights[key]
    title     = h.get("Title", "Untitled Highlight")
    blurb     = process_description(h.get("Blurb", ""), limit=500)
    link      = h.get("ShowRelatedURL") or "https://gitlab.com/WeSuckLess/Reactor"
    atom_id   = h.get("AtomID")
    image_url = h.get("ImageURL")
    atom_name = h.get("AtomName")

    if atom_id:
        blurb += f" [Atom: {atom_id}]"

    item = ET.SubElement(channel, "item")
    ET.SubElement(item, "title").text = f"[Highlight] {title}"
    # Wrap description content with a CDATA placeholder
    ET.SubElement(item, "description").text = f"__CDATA_START__{blurb}__CDATA_END__"
    ET.SubElement(item, "link").text = link
    ET.SubElement(item, "guid", isPermaLink="false").text = f"highlight-{key}-{atom_id or title}"
    ET.SubElement(item, "pubDate").text = rfc822(now - timedelta(minutes=idx))
    ET.SubElement(item, "category").text = "Highlight"

    # Use AtomName as creator for highlights, default to 'Reactor'
    creator_name = atom_name if atom_name else "Reactor"
    ET.SubElement(item, "{http://purl.org/dc/elements/1.1/}creator").text = creator_name

    # Include the ImageURL as Media RSS element structures
    if image_url:
        media_content = ET.SubElement(item, "{http://search.yahoo.com/mrss/}content", **{
            "height": "192",
            "medium": "image",
            "type": "image/png",
            "url": image_url,
            "width": "256"
        })
        ET.SubElement(media_content, "{http://search.yahoo.com/mrss/}thumbnail", **{
            "height": "192",
            "url": image_url,
            "width": "256"
        })

# --- Parse and Sort Packages by Date desc across all Repositories ---
packages = []
repo_data = json.loads(RepoJSON)

for repo in repo_data["Repos"]:
    repo_name = repo["Name"]
    repo_url = repo["JSON"]
    print(f"⌒ Fetching {repo_name} packages…")
    try:
        # Ensure we target the direct raw content raw files on Gitlab instead of blob viewer pages
        raw_url = repo_url.replace("/blob/", "/raw/")
        r_json = fetch_json(raw_url)
        print(f" {len(r_json)} entries loaded")
        for key, p in r_json.items():
            if p.get("__ctor") != "Atom":
                continue
            packages.append((repo_name, key, p, parse_reactor_date(p.get("Date", {}))))
    except Exception as e:
        print(f" Failed to load {repo_name} repository packages: {e}")

packages.sort(key=lambda x: x[3], reverse=True)

# Include only the newest packages
for repo_name, key, p, dt in packages[:MAX_PACKAGES]:
    name     = p.get("Name", "Unnamed")
    author   = p.get("Author", "Unknown")
    version  = p.get("Version", "")
    category_str = p.get("Category", "")
    # Retrieve full description with full HTML tags unescaped and un-truncated
    desc     = process_description(p.get("Description", ""), limit=None)
    zip_link = p.get("Zipfile", "https://gitlab.com/WeSuckLess/Reactor")

    # Format hierarchical category structure using the repo's specific name prefix
    formatted_title = f"{repo_name}/{category_str}/{name}" if category_str else f"{repo_name}/{name}"

    item = ET.SubElement(channel, "item")
    ET.SubElement(item, "title").text = formatted_title

    package_desc = (
        f"<b>Author:</b> {author}<br/>"
        f"<b>Version:</b> {version}<br/><br/>{desc}"
    )
    # Wrap description content with a CDATA placeholder
    ET.SubElement(item, "description").text = f"__CDATA_START__{package_desc}__CDATA_END__"
    ET.SubElement(item, "link").text = zip_link
    ET.SubElement(item, "guid", isPermaLink="true").text = zip_link
    ET.SubElement(item, "pubDate").text = rfc822(dt)

    # Use Author as creator for packages, default to 'Reactor'
    creator_name = author if author else "Reactor"
    ET.SubElement(item, "{http://purl.org/dc/elements/1.1/}creator").text = creator_name

    if category_str:
        categories = [c.strip() for c in category_str.split("/") if c.strip()]
        for cat in categories:
            ET.SubElement(item, "category").text = cat

rough  = ET.tostring(rss, encoding="utf-8")
pretty = minidom.parseString(rough).toprettyxml(indent="  ", encoding="utf-8")
pretty_str = pretty.decode("utf-8")
def unescape_cdata_blocks(match):
    inner_content = match.group(1)
    inner_content = inner_content.replace("&lt;", "<").replace("&gt;", ">").replace("&amp;", "&")
    return f"<![CDATA[{inner_content}]]>"

# Push HTML Description into CDATA block
pretty_str = re.sub(r"__CDATA_START__(.*?)__CDATA_END__", unescape_cdata_blocks, pretty_str, flags=re.DOTALL)
pretty = pretty_str.encode("utf-8")

# Write XML output
with open(OUTPUT_FILE, "wb") as f:
    f.write(pretty)

size_kb = round(len(pretty) / 1024, 1)
print(f"\n RSS feed updated successfully: {OUTPUT_FILE} ({size_kb} KB)")

# Download the 'feed.xml' file
try:
    files.download('feed.xml')
    print("Download prompt opened successfully!")
except Exception as e:
    print(f"Could not trigger automatic download: {e}")
    print("You can still download it manually from the Files folder in the left sidebar.")

In [ ]:
import xml.etree.ElementTree as ET

# Parse the generated XML to validate ity
try:
    tree = ET.parse("feed.xml")
    root = tree.getroot()
    print(" XML parsed successfully!")

    channel = root.find("channel")
    items = channel.findall("item")
    print(f"Total items found in feed: {len(items)}")

    for idx, item in enumerate(items[:3]):
        title = item.find("title").text
        desc = item.find("description").text
        print(f"\n[Item {idx+1}] Title: {title}")
        print(f"Description Snippet: {desc[:150]}...")

    with open("feed.xml", "r", encoding="utf-8") as f:
        raw_xml = f.read()

    if "<![CDATA[" in raw_xml:
        print("\n SUCCESS: Raw '<![CDATA[' tags were verified within the file!")
    else:
        print("\n ERROR: CDATA tags were not found in the raw XML output!")

except Exception as e:
    print(f" XML validation failed: {e}")
